# Notebook 02: Autograd Internals

## Why This Matters

Most deep learning bugs that survive into production are gradient bugs -- they do not crash, they just quietly produce worse models. Understanding how autograd builds and traverses the computation graph lets you confidently debug NaN gradients, catch double-backprop mistakes, and safely implement gradient accumulation, MAML meta-learning, and differentiable physics. Engineers who treat autograd as a black box eventually hit a wall they cannot debug.

## Background

PyTorch's autograd engine implements **reverse-mode automatic differentiation** (backpropagation). During the forward pass each operation creates a `Function` node that stores:
- Inputs needed for the backward computation (via `ctx.save_for_backward`)
- References to the next functions in the graph (`next_functions`)

During `backward()`, the engine traverses this DAG in topological order, calling each `Function`'s `backward` static method and accumulating gradients using the chain rule:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$$

For a composed function $L = f(g(x))$:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial f} \cdot \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
print("Ready.")

## 1. How the Graph Is Built During the Forward Pass

Every time you apply an operation to a `requires_grad` tensor, PyTorch attaches a `grad_fn` to the result that points back to the operation. This forms the edges of the DAG. The leaf tensors (parameters) are the sources; the loss is the sink. `backward()` walks from sink to sources.

In [ ]:
# Build a tiny graph and inspect it
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
W = torch.tensor([[0.1, 0.2, 0.3],
                  [0.4, 0.5, 0.6]], requires_grad=True)

# forward pass
h = W @ x           # MmBackward0
h_act = h.relu()    # ReluBackward0
loss = h_act.sum()  # SumBackward0

print("Computation graph (grad_fn chain):")
print(f"  loss.grad_fn:       {loss.grad_fn}")
print(f"  h_act.grad_fn:      {h_act.grad_fn}")
print(f"  h.grad_fn:          {h.grad_fn}")
print(f"  x.grad_fn:          {x.grad_fn}  (leaf has no grad_fn)")

# inspect next_functions -- the edges
print("\nEdges from SumBackward0:")
for fn, idx in loss.grad_fn.next_functions:
    print(f"  -> {fn}")
print("\nEdges from ReluBackward0:")
for fn, idx in h_act.grad_fn.next_functions:
    print(f"  -> {fn}")

# do backward
loss.backward()
print(f"\nx.grad:  {x.grad}")
print(f"W.grad:\n{W.grad}")

## 2. Gradient Accumulation -- The Critical Pattern

`backward()` **accumulates** gradients into `.grad` (adds to whatever is already there). This is intentional: it lets you simulate large batch sizes by running multiple smaller forward/backward passes before calling `optimizer.step()`. But it means you **must** call `optimizer.zero_grad()` (or `zero_grad(set_to_none=True)`) before each logical batch, or your gradients will snowball across iterations.

The accumulation pattern is the standard solution when a model fits on one GPU but the desired batch size does not.

In [ ]:
# Demonstrate accumulation
x = torch.tensor([2.0], requires_grad=True)
optimizer = torch.optim.SGD([x], lr=0.1)

print("=== Accumulation bug (forgot zero_grad) ===")
for step in range(3):
    loss = x ** 2
    loss.backward()
    print(f"  step {step}: x.grad = {x.grad.item():.2f}  (should be 4.0 every time)")
# Gradients pile up: 4, 8, 12 instead of 4, 4, 4

print()
print("=== Correct: zero_grad each step ===")
x = torch.tensor([2.0], requires_grad=True)
optimizer = torch.optim.SGD([x], lr=0.1)
for step in range(3):
    optimizer.zero_grad()
    loss = x ** 2
    loss.backward()
    print(f"  step {step}: x.grad = {x.grad.item():.2f}  (correct: always 4.0)")
    optimizer.step()

print()
print("=== Gradient accumulation for large effective batch ===")
x = torch.tensor([2.0], requires_grad=True)
optimizer = torch.optim.SGD([x], lr=0.1)
ACCUM_STEPS = 4  # effective batch = 4 x micro_batch
optimizer.zero_grad()
for micro_step in range(ACCUM_STEPS):
    micro_loss = (x ** 2) / ACCUM_STEPS   # divide to normalise
    micro_loss.backward()
    print(f"  micro_step {micro_step}: x.grad accumulated = {x.grad.item():.4f}")
print(f"Final accumulated grad: {x.grad.item():.4f}  (same as single step = 4.0)")
optimizer.step()

## 3. retain_graph and create_graph

By default PyTorch **frees the computation graph** immediately after `backward()` completes -- the intermediate activations stored in `grad_fn` nodes are released. If you need to call `backward()` again on the same graph (e.g., for diagnostics or RL policy gradients), pass `retain_graph=True`.

`create_graph=True` is different and more expensive: it tells autograd to build a graph *of the backward pass itself*, enabling higher-order derivatives. You need this for MAML (gradient-through-gradient), computing the Hessian, or any technique requiring `grad` of `grad`.

In [ ]:
# retain_graph
x = torch.tensor([3.0], requires_grad=True)
y = x ** 3  # dy/dx = 3x^2 = 27

y.backward(retain_graph=True)
print(f"First backward:  x.grad = {x.grad.item()}")   # 27

x.grad.zero_()
y.backward(retain_graph=False)  # graph freed now
print(f"Second backward: x.grad = {x.grad.item()}")   # still 27

# create_graph=True -- enables d^2y/dx^2
x = torch.tensor([3.0], requires_grad=True)
y = x ** 3

# First derivative (with graph so we can differentiate again)
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"\ndy/dx at x=3:   {grad1.item():.1f}  (expected: 27.0)")

# Second derivative
grad2 = torch.autograd.grad(grad1, x)[0]
print(f"d^2y/dx^2 at x=3: {grad2.item():.1f}  (expected: 18.0)")  # 6x = 18

## 4. Custom autograd.Function

Sometimes you need a differentiable operation whose backward pass is not automatically derived -- for example, a custom CUDA kernel, a numerically stable custom activation, or a straight-through estimator for discrete operations.

The pattern: subclass `torch.autograd.Function`, implement `forward` and `backward` as `@staticmethod` methods, and use `ctx.save_for_backward` / `ctx.saved_tensors` to pass tensors from forward to backward.

In [ ]:
class ClampedSigmoid(torch.autograd.Function):
    # Sigmoid clamped to [eps, 1-eps]: numerically safer for extreme inputs.

    @staticmethod
    def forward(ctx, x, eps=1e-6):
        sig = torch.sigmoid(x)
        sig_clamped = sig.clamp(eps, 1 - eps)
        ctx.save_for_backward(sig_clamped)
        return sig_clamped

    @staticmethod
    def backward(ctx, grad_output):
        (sig,) = ctx.saved_tensors
        # d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x))
        grad_input = grad_output * sig * (1 - sig)
        return grad_input, None  # None for eps (not a tensor)

clamped_sigmoid = ClampedSigmoid.apply

# test forward
x = torch.tensor([-10.0, 0.0, 10.0], requires_grad=True)
y = clamped_sigmoid(x)
print(f"Input:  {x.detach().tolist()}")
print(f"Output: {[f'{v:.6f}' for v in y.tolist()]}")

# test backward
y.sum().backward()
print(f"Grads:  {x.grad.tolist()}")

# gradient check -- compares numerical vs analytical gradients
x_check = torch.randn(5, requires_grad=True, dtype=torch.float64)
result = torch.autograd.gradcheck(
    lambda t: ClampedSigmoid.apply(t, 1e-6),
    x_check,
    eps=1e-6,
    atol=1e-4
)
print(f"\ngradcheck passed: {result}")

## 5. torch.autograd.grad vs .backward()

`.backward()` populates `.grad` attributes on leaf tensors. It is the standard training interface.

`torch.autograd.grad()` returns gradient tensors directly without touching `.grad`. It is the right choice when:
- You need gradients w.r.t. non-leaf tensors
- You need per-sample gradients (in combination with `vmap`)
- You want gradient computation to be functional (no side effects on `.grad`)
- You are building a custom optimizer or meta-learning loop

In [ ]:
# .backward() vs autograd.grad()
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (x ** 2).sum()

# Method 1: backward populates x.grad (in-place)
loss.backward()
print(f"Via backward():      x.grad = {x.grad.tolist()}")

# Method 2: autograd.grad returns tensors directly
x2 = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss2 = (x2 ** 2).sum()
grads = torch.autograd.grad(loss2, x2)
print(f"Via autograd.grad(): grads  = {grads[0].tolist()}")
print(f"                     x2.grad = {x2.grad}")  # None! no side effect

# gradient w.r.t. intermediate tensor (impossible with .backward() alone)
x3 = torch.tensor([2.0], requires_grad=True)
h  = x3 * 3              # intermediate: h = 3x
loss3 = h ** 2            # loss = 9x^2  ->  dloss/dh = 2h = 12
grad_h = torch.autograd.grad(loss3, h)[0]
print(f"\ndloss/dh (intermediate grad): {grad_h.item()}  (expected: 12.0)")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| Graph construction | Built dynamically during forward pass; each op creates a Function node |
| backward() | Traverses DAG in topological order, calls each node's backward method |
| Gradient accumulation | `.grad` adds on each backward(); must zero_grad() per logical batch |
| retain_graph | Keep graph after backward() for multiple backward calls |
| create_graph | Build graph of backward pass itself; enables higher-order gradients |
| autograd.Function | Custom differentiable op with explicit forward/backward; ctx.save_for_backward |
| autograd.grad() | Functional gradient computation; no .grad side effects; works on non-leaves |

### Common Pitfalls
- Forgetting `zero_grad()` -- gradients accumulate across steps, inflating parameter updates
- Calling `backward()` twice without `retain_graph=True` -- RuntimeError on second call
- Not dividing loss by `ACCUM_STEPS` in gradient accumulation -- effective LR scales up
- Using `ctx.save_for_backward` for non-tensors -- use `ctx.<attr>` for scalars/metadata
- Calling `autograd.grad` without `create_graph=True` when you need higher-order derivs

### Next: Notebook 03 -- nn.Module Patterns